In [12]:
"""
Extrae las tablas de Caudal Promedio Diario de TODOS los años
presentes en los informes HTML de la CVC y las guarda como hojas
separadas dentro de un único libro Excel por estación.

ACTUALIZADO: ahora recorre TODAS las carpetas con Informe.html
automáticamente — no hace falta editar nada al agregar carpetas nuevas.

Carpetas sin datos de caudal en el HTML (solo informe de calidad/sin tablas)
se omiten automáticamente si no se encuentran años en el HTML.

Requiere:
    pip install beautifulsoup4 lxml openpyxl
"""

from pathlib import Path
from bs4 import BeautifulSoup
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── Carpetas a omitir — solo tienen informe de calidad, sin tablas de caudal ──
OMITIR = {
    "CAJIBIO",
    "CAUCA - SUAREZ (SIN DATOS CAUDAL)",
    "NAVARRO (SIN DATOS CAUDAL)",
    "LAS VEGAS (S)",
    "MICHICAL (S)",
    "PASO DEL COMERCIO (S)",
    "PUERTO ISAACS (S)",
}

MESES = ["ENE","FEB","MAR","ABR","MAY","JUN","JUL","AGO","SEP","OCT","NOV","DIC"]
STOP  = {"MÁXIMO","MAXIMO","MÍNIMO","MINIMO","MEDIO"}


def limpiar(v):
    v = v.strip()
    if v in ("***<", "***", "<", "", "—", "-"):
        return None
    try:
        return float(v.replace(",", "."))
    except ValueError:
        return None


def extraer_año(spans, data_start, data_end):
    block = spans[data_start:data_end]
    max_off = next(
        (i for i, t in enumerate(block) if t.upper() in ("MÁXIMO","MAXIMO")),
        len(block)
    )
    datos = block[:max_off]
    filas = {}
    i = 0
    while i < len(datos):
        tok = datos[i].strip()
        if tok.upper() in STOP:
            i += 1
            continue
        try:
            dia = int(tok)
            if not 1 <= dia <= 31:
                i += 1
                continue
        except ValueError:
            i += 1
            continue
        vals = []
        j = i + 1
        while j < len(datos):
            candidate = datos[j].strip()
            if candidate.upper() in STOP:
                break
            try:
                next_dia = int(candidate)
                if 1 <= next_dia <= 31 and len(vals) > 0:
                    break
            except ValueError:
                pass
            is_val = False
            try:
                float(candidate.replace(",", "."))
                is_val = True
            except ValueError:
                if candidate in ("***<", "***", "<"):
                    is_val = True
            if is_val:
                vals.append(limpiar(candidate))
                j += 1
            else:
                j += 1
        filas[dia] = (vals + [None] * 12)[:12]
        i = j
    rows = [[dia] + filas[dia] for dia in sorted(filas.keys())]
    max_vals = [limpiar(v) for v in block[max_off + 1 : max_off + 13]]
    max_vals = (max_vals + [None] * 12)[:12]
    min_off = next(
        (i for i, t in enumerate(block) if t.upper() in ("MÍNIMO","MINIMO")), None
    )
    min_vals = [None] * 12
    if min_off is not None:
        raw = block[min_off + 1 : min_off + 30]
        tmp = []
        k = 0
        while len(tmp) < 12 and k < len(raw) - 1:
            try:
                tmp.append(float(raw[k].replace(",", ".")))
                k += 2
            except Exception:
                k += 1
        min_vals = (tmp + [None] * 12)[:12]
    med_off = next(
        (i for i, t in enumerate(block) if t.upper() == "MEDIO"), None
    )
    med_vals = [None] * 12
    if med_off is not None:
        med_vals = [limpiar(v) for v in block[med_off + 1 : med_off + 13]]
        med_vals = (med_vals + [None] * 12)[:12]
    return rows, max_vals, min_vals, med_vals


def escribir_hoja(ws, estacion, año, rows, max_vals, min_vals, med_vals):
    borde = Border(
        left=Side(style="thin"), right=Side(style="thin"),
        top=Side(style="thin"), bottom=Side(style="thin"),
    )
    ws.merge_cells("A1:M1")
    c = ws["A1"]
    c.value = (
        f"Caudal Promedio Diario  |  Estación: CAUCA - {estacion}  "
        f"|  Año: {año}  |  Unidad: m³/s"
    )
    c.font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
    c.fill = PatternFill("solid", fgColor="1D4E8F")
    c.alignment = Alignment(horizontal="center", vertical="center")
    ws.row_dimensions[1].height = 20
    for col, h in enumerate(["DÍA"] + MESES, 1):
        c = ws.cell(row=2, column=col, value=h)
        c.font = Font(bold=True, color="FFFFFF", name="Arial", size=9)
        c.fill = PatternFill("solid", fgColor="2E75B6")
        c.alignment = Alignment(horizontal="center")
        c.border = borde
    for r, row in enumerate(rows, 3):
        for col, val in enumerate(row, 1):
            c = ws.cell(row=r, column=col)
            if col == 1:
                c.value = int(val)
                c.fill = PatternFill("solid", fgColor="EBF3FB")
                c.alignment = Alignment(horizontal="center")
                c.font = Font(bold=True, name="Arial", size=9)
            elif val is None:
                c.value = "S/D"
                c.font = Font(name="Arial", size=9, italic=True, color="999999")
                c.alignment = Alignment(horizontal="center")
            else:
                c.value = val
                c.number_format = "#,##0.00"
                c.font = Font(name="Arial", size=9)
                c.alignment = Alignment(horizontal="right")
            c.border = borde
    sep = len(rows) + 3
    for offset, (etiq, vals, bg) in enumerate([
        ("MÁXIMO", max_vals, "FFC000"),
        ("MÍNIMO", min_vals, "70AD47"),
        ("MEDIO",  med_vals, "4472C4"),
    ]):
        r = sep + offset
        c = ws.cell(row=r, column=1, value=etiq)
        c.font = Font(bold=True, color="FFFFFF", name="Arial", size=9)
        c.fill = PatternFill("solid", fgColor=bg)
        c.alignment = Alignment(horizontal="center")
        c.border = borde
        for col, v in enumerate(vals, 2):
            cell = ws.cell(row=r, column=col, value=v)
            cell.font = Font(bold=True, name="Arial", size=9, color="FFFFFF")
            cell.fill = PatternFill("solid", fgColor=bg)
            cell.alignment = Alignment(horizontal="right")
            if v is not None:
                cell.number_format = "#,##0.00"
            cell.border = borde
    ws.column_dimensions["A"].width = 6
    for col in range(2, 14):
        ws.column_dimensions[get_column_letter(col)].width = 10
    ws.freeze_panes = "B3"


def procesar_carpeta(carpeta: Path):
    html_file = carpeta / "Informe.html"
    if not html_file.exists():
        return
    estacion = carpeta.name.upper()
    if estacion in OMITIR:
        print(f"  ⏭️  {carpeta.name} — omitida (sin datos de caudal)")
        return
    output_file = carpeta / "caudal_2010.xlsx"
    print(f"📄 Procesando {carpeta.name} ...")
    with open(html_file, encoding="utf-8", errors="replace") as f:
        soup = BeautifulSoup(f, "lxml")
    spans = [s.get_text(strip=True) for s in soup.find_all("span")]
    año_indices = [
        (i, spans[i + 1])
        for i, t in enumerate(spans)
        if t == "AÑO" and i + 1 < len(spans) and spans[i + 1].isdigit()
    ]
    if not año_indices:
        print(f"  ⚠️  Sin tablas de caudal en el HTML — omitida")
        return
    print(f"   → {len(año_indices)} años: {', '.join(a for _, a in año_indices)}")
    wb = Workbook()
    wb.remove(wb.active)
    for k, (idx, año) in enumerate(año_indices):
        data_start = idx + 15
        data_end   = año_indices[k + 1][0] if k + 1 < len(año_indices) else len(spans)
        rows, max_vals, min_vals, med_vals = extraer_año(spans, data_start, data_end)
        ws = wb.create_sheet(title=str(año))
        escribir_hoja(ws, estacion, año, rows, max_vals, min_vals, med_vals)
        print(f"   ✔  {año}  —  {len(rows)} días")
    wb.save(output_file)
    print(f"   ✅ Guardado: {output_file}")


def main():
    raiz = Path(".")
    carpetas = sorted([d for d in raiz.iterdir() if d.is_dir()])
    print(f"🗂  {len(carpetas)} carpetas encontradas")
    for carpeta in carpetas:
        procesar_carpeta(carpeta)
    print("✅ Proceso completado.")


if __name__ == "__main__":
    main()


🗂  26 carpetas encontradas
📄 Procesando ANACARO ...
   → 17 años: 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026
   ✔  2010  —  31 días
   ✔  2011  —  31 días
   ✔  2012  —  31 días
   ✔  2013  —  31 días
   ✔  2014  —  31 días
   ✔  2015  —  31 días
   ✔  2016  —  31 días
   ✔  2017  —  31 días
   ✔  2018  —  31 días
   ✔  2019  —  31 días
   ✔  2020  —  31 días
   ✔  2021  —  31 días
   ✔  2022  —  31 días
   ✔  2023  —  31 días
   ✔  2024  —  31 días
   ✔  2025  —  31 días
   ✔  2026  —  31 días
   ✅ Guardado: ANACARO\caudal_2010.xlsx
📄 Procesando GARZONERO NORTE ...
   → 17 años: 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026
   ✔  2010  —  31 días
   ✔  2011  —  31 días
   ✔  2012  —  31 días
   ✔  2013  —  31 días
   ✔  2014  —  31 días
   ✔  2015  —  31 días
   ✔  2016  —  31 días
   ✔  2017  —  31 días
   ✔  2018  —  31 días
   ✔  2019  —  31 días
   ✔  2020  —  31 días
 

In [13]:
"""
Genera la gráfica de Caudal Promedio Anual (2011-2026)
para cada estación y la guarda dentro de su carpeta.

Estructura esperada:
    <raíz>/
    ├── ANACARO/
    │   └── caudal_2010.xlsx
    ├── CAJIBIO/
    │   └── caudal_2010.xlsx
    ├── ...
    └── graficar_caudal.py   ← este script

Uso:
    python graficar_caudal.py

Requiere:
    pip install openpyxl matplotlib seaborn
"""

from pathlib import Path
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from openpyxl import load_workbook

warnings.filterwarnings("ignore")

# ── Configuración ──────────────────────────────────────────────
AÑO_INICIO = 2011
AÑO_FIN    = 2026
EXCEL_NAME = "caudal_2010.xlsx"
DPI        = 150
# ───────────────────────────────────────────────────────────────


def leer_promedios_anuales(excel_path: Path) -> dict:
    """
    Lee cada hoja (un año) y extrae el promedio anual
    a partir de la fila MEDIO del Excel.
    Devuelve {año: promedio_anual} para años en rango.
    """
    wb = load_workbook(excel_path, data_only=True)
    resultados = {}

    for sheet_name in wb.sheetnames:
        try:
            año = int(sheet_name)
        except ValueError:
            continue
        if not (AÑO_INICIO <= año <= AÑO_FIN):
            continue

        ws = wb[sheet_name]
        for row in ws.iter_rows(min_row=3, values_only=True):
            if row[0] is None:
                continue
            if str(row[0]).strip().upper() == "MEDIO":
                vals = [v for v in row[1:13] if isinstance(v, (int, float))]
                if vals:
                    resultados[año] = round(np.mean(vals), 2)
                break

    return resultados


def graficar_promedio_anual(estacion: str, promedios: dict, out_path: Path):
    """Genera y guarda la gráfica de barras de caudal promedio anual."""

    años    = sorted(promedios.keys())
    valores = [promedios[a] for a in años]

    # Paleta degradada de azul a verde agua según magnitud
    norm    = plt.Normalize(min(valores), max(valores))
    colores = plt.cm.YlGnBu(norm(valores))

    fig, ax = plt.subplots(figsize=(13, 6))
    fig.patch.set_facecolor("#F7F9FC")
    ax.set_facecolor("#F7F9FC")

    bars = ax.bar(años, valores, color=colores, edgecolor="white",
                  linewidth=0.7, width=0.65, zorder=3)

    # Valor sobre cada barra
    for bar, val in zip(bars, valores):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(valores) * 0.012,
            f"{val:,.0f}",
            ha="center", va="bottom",
            fontsize=8, fontweight="bold", color="#333333"
        )

    # Línea de promedio general
    prom_total = np.mean(valores)
    ax.axhline(prom_total, color="#E63946", linewidth=1.4,
               linestyle="--", zorder=4,
               label=f"Promedio período: {prom_total:,.0f} m³/s")

    # Formato de ejes
    ax.set_xticks(años)
    ax.set_xticklabels([str(a) for a in años], rotation=45,
                       ha="right", fontsize=9)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.tick_params(axis="y", labelsize=9)

    # Grilla y bordes
    ax.yaxis.grid(True, linestyle="--", alpha=0.45, zorder=0)
    ax.set_axisbelow(True)
    sns.despine(ax=ax, left=False, bottom=False)

    # Títulos y etiquetas
    ax.set_title(
        f"Caudal Promedio Anual — Estación {estacion}",
        fontsize=13, fontweight="bold", pad=14, color="#1A1A2E"
    )
    ax.set_ylabel("Caudal promedio (m³/s)", fontsize=10, labelpad=8)
    ax.set_xlabel("Año", fontsize=10, labelpad=8)
    ax.legend(fontsize=9, framealpha=0.7)

    fig.tight_layout()
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close(fig)


def main():
    raiz = Path(".")

    estaciones = sorted([
        d for d in raiz.iterdir()
        if d.is_dir() and (d / EXCEL_NAME).exists()
    ])

    if not estaciones:
        print("❌ No se encontraron carpetas con 'caudal_2010.xlsx'.")
        print("   Corre el script desde la raíz del proyecto.")
        return

    print(f"🗂  {len(estaciones)} estaciones encontradas\n")

    for carpeta in estaciones:
        promedios = leer_promedios_anuales(carpeta / EXCEL_NAME)

        if not promedios:
            print(f"  ⚠  {carpeta.name:20s} — sin datos en {AÑO_INICIO}-{AÑO_FIN}")
            continue

        out_path = carpeta / "promedio_anual.png"
        graficar_promedio_anual(carpeta.name.title(), promedios, out_path)

        años = sorted(promedios.keys())
        print(f"  ✔  {carpeta.name:20s} — {años[0]}–{años[-1]}"
              f"  →  {out_path}")

    print("\n✅ Listo.")


if __name__ == "__main__":
    main()

🗂  18 estaciones encontradas

  ✔  ANACARO              — 2011–2026  →  ANACARO\promedio_anual.png
  ✔  GARZONERO NORTE      — 2011–2026  →  GARZONERO NORTE\promedio_anual.png
  ✔  GARZONERO SUR (S - 2023) — 2015–2026  →  GARZONERO SUR (S - 2023)\promedio_anual.png
  ✔  GUAYABAL (HASTA 2022) — 2011–2026  →  GUAYABAL (HASTA 2022)\promedio_anual.png
  ✔  HORMIGUERO           — 2011–2026  →  HORMIGUERO\promedio_anual.png
  ✔  JUAN DIAZ            — 2011–2026  →  JUAN DIAZ\promedio_anual.png
  ✔  JUANCHITO (S HASTA 2017) — 2015–2026  →  JUANCHITO (S HASTA 2017)\promedio_anual.png
  ✔  LA BALSA             — 2011–2026  →  LA BALSA\promedio_anual.png
  ✔  LA BOLSA             — 2015–2026  →  LA BOLSA\promedio_anual.png
  ✔  LA FLORESTA          — 2011–2026  →  LA FLORESTA\promedio_anual.png
  ✔  LA PRIMAVERA         — 2015–2026  →  LA PRIMAVERA\promedio_anual.png
  ✔  LA VICTORIA          — 2011–2026  →  LA VICTORIA\promedio_anual.png
  ✔  MEDICANOA            — 2011–2026  →  MEDICANOA\prome

In [14]:
"""
Genera una gráfica de Caudal Promedio Mensual por quinquenio
para cada estación y la guarda dentro de su carpeta.

Quinquenios:
    Q1: 2010 – 2014
    Q2: 2015 – 2019
    Q3: 2020 – 2026

Estructura esperada:
    <raíz>/
    ├── ANACARO/
    │   └── caudal_2010.xlsx
    ├── CAJIBIO/
    │   └── caudal_2010.xlsx
    └── graficar_quinquenios.py   ← este script

Uso:
    python graficar_quinquenios.py

Requiere:
    pip install openpyxl matplotlib seaborn numpy
"""

from pathlib import Path
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from openpyxl import load_workbook

warnings.filterwarnings("ignore")

# ── Configuración ──────────────────────────────────────────────
EXCEL_NAME = "caudal_2010.xlsx"
DPI        = 150

QUINQUENIOS = {
    "2010 – 2014": range(2010, 2015),
    "2015 – 2019": range(2015, 2020),
    "2020 – 2026": range(2020, 2027),
}

COLORES = {
    "2010 – 2014": "#2563EB",   # azul
    "2015 – 2019": "#16A34A",   # verde
    "2020 – 2026": "#DC2626",   # rojo
}

MESES     = ["ENE","FEB","MAR","ABR","MAY","JUN",
             "JUL","AGO","SEP","OCT","NOV","DIC"]
# ───────────────────────────────────────────────────────────────


def leer_medios(excel_path: Path) -> dict:
    """
    Lee cada hoja y devuelve {año: [prom_ene, prom_feb, ..., prom_dic]}
    usando la fila MEDIO de cada hoja.
    """
    wb = load_workbook(excel_path, data_only=True)
    datos = {}

    for sheet_name in wb.sheetnames:
        try:
            año = int(sheet_name)
        except ValueError:
            continue

        ws = wb[sheet_name]
        for row in ws.iter_rows(min_row=3, values_only=True):
            if row[0] is None:
                continue
            if str(row[0]).strip().upper() == "MEDIO":
                vals = [
                    v if isinstance(v, (int, float)) else None
                    for v in row[1:13]
                ]
                datos[año] = vals
                break

    return datos


def calcular_promedio_quinquenio(datos: dict, años: range) -> list:
    """
    Promedia los valores mensuales de todos los años del quinquenio.
    Ignora None (S/D) en el promedio.
    Devuelve lista de 12 valores (None si no hay dato en ese mes).
    """
    result = []
    for mes_idx in range(12):
        vals = [
            datos[a][mes_idx]
            for a in años
            if a in datos and datos[a][mes_idx] is not None
        ]
        result.append(round(np.mean(vals), 2) if vals else None)
    return result


def graficar_quinquenios(estacion: str, datos: dict, out_path: Path):
    """Genera y guarda la gráfica de líneas por quinquenio."""

    fig, ax = plt.subplots(figsize=(13, 6))
    fig.patch.set_facecolor("#F7F9FC")
    ax.set_facecolor("#F7F9FC")

    tiene_datos = False

    for label, años in QUINQUENIOS.items():
        prom = calcular_promedio_quinquenio(datos, años)

        # Coordenadas solo donde hay valor
        xs = [i for i, v in enumerate(prom) if v is not None]
        ys = [v for v in prom if v is not None]

        if not ys:
            continue

        tiene_datos = True
        color = COLORES[label]

        ax.plot(
            xs, ys,
            color=color, linewidth=2.2, marker="o",
            markersize=6, markerfacecolor="white",
            markeredgewidth=2, markeredgecolor=color,
            label=label, zorder=4
        )

        # Etiqueta de valor sobre cada punto
        for x, y in zip(xs, ys):
            ax.annotate(
                f"{y:,.0f}",
                xy=(x, y),
                xytext=(0, 9),
                textcoords="offset points",
                ha="center", va="bottom",
                fontsize=7, color=color, fontweight="bold"
            )

    if not tiene_datos:
        plt.close(fig)
        return

    # Eje X — meses
    ax.set_xticks(range(12))
    ax.set_xticklabels(MESES, fontsize=10)

    # Eje Y
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.tick_params(axis="y", labelsize=9)

    # Grilla
    ax.yaxis.grid(True, linestyle="--", alpha=0.4, zorder=0)
    ax.set_axisbelow(True)
    sns.despine(ax=ax)

    # Títulos
    ax.set_title(
        f"Caudal Promedio Mensual por Quinquenio — Estación {estacion}",
        fontsize=13, fontweight="bold", pad=14, color="#1A1A2E"
    )
    ax.set_ylabel("Caudal promedio (m³/s)", fontsize=10, labelpad=8)
    ax.set_xlabel("Mes", fontsize=10, labelpad=8)

    ax.legend(
        title="Quinquenio", fontsize=10, title_fontsize=10,
        framealpha=0.8, loc="upper right"
    )

    fig.tight_layout()
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close(fig)


def procesar_estacion(carpeta: Path):
    excel = carpeta / EXCEL_NAME
    if not excel.exists():
        print(f"  ⚠  {carpeta.name:22s} — sin Excel")
        return

    datos = leer_medios(excel)
    if not datos:
        print(f"  ⚠  {carpeta.name:22s} — sin datos")
        return

    out_path = carpeta / "quinquenios_caudal.png"
    graficar_quinquenios(carpeta.name.title(), datos, out_path)
    print(f"  ✔  {carpeta.name:22s} → {out_path}")


def main():
    raiz = Path(".")

    estaciones = sorted([
        d for d in raiz.iterdir()
        if d.is_dir() and (d / EXCEL_NAME).exists()
    ])

    if not estaciones:
        print("❌ No se encontraron carpetas con 'caudal_2010.xlsx'.")
        print("   Corre el script desde la raíz del proyecto.")
        return

    print(f"🗂  {len(estaciones)} estaciones encontradas\n")
    for carpeta in estaciones:
        procesar_estacion(carpeta)

    print("\n✅ Listo. Gráficas guardadas como 'quinquenios_caudal.png' "
          "en cada carpeta.")


if __name__ == "__main__":
    main()

🗂  18 estaciones encontradas

  ✔  ANACARO                → ANACARO\quinquenios_caudal.png
  ✔  GARZONERO NORTE        → GARZONERO NORTE\quinquenios_caudal.png
  ✔  GARZONERO SUR (S - 2023) → GARZONERO SUR (S - 2023)\quinquenios_caudal.png
  ✔  GUAYABAL (HASTA 2022)  → GUAYABAL (HASTA 2022)\quinquenios_caudal.png
  ✔  HORMIGUERO             → HORMIGUERO\quinquenios_caudal.png
  ✔  JUAN DIAZ              → JUAN DIAZ\quinquenios_caudal.png
  ✔  JUANCHITO (S HASTA 2017) → JUANCHITO (S HASTA 2017)\quinquenios_caudal.png
  ✔  LA BALSA               → LA BALSA\quinquenios_caudal.png
  ✔  LA BOLSA               → LA BOLSA\quinquenios_caudal.png
  ✔  LA FLORESTA            → LA FLORESTA\quinquenios_caudal.png
  ✔  LA PRIMAVERA           → LA PRIMAVERA\quinquenios_caudal.png
  ✔  LA VICTORIA            → LA VICTORIA\quinquenios_caudal.png
  ✔  MEDICANOA              → MEDICANOA\quinquenios_caudal.png
  ✔  PAN DE AZUCAR          → PAN DE AZUCAR\quinquenios_caudal.png
  ✔  PASO LA TORRE          

### Debo verificar los rangos de Caudal con los del PMC para verificar el tema de las gráficas.

In [15]:
"""
Cruza datos de calidad del agua con caudal diario del río Cauca,
clasifica cada monitoreo como Verano / Transición / Invierno
usando los rangos del informe PMC (Univ. del Valle; CVC 2001),
y guarda un Excel con los resultados enriquecidos.

Estructura esperada:
    <raíz>/
    ├── ANACARO/
    │   └── caudal_2010.xlsx
    ├── LA VICTORIA/
    │   └── caudal_2010.xlsx
    ├── ...
    ├── Calidad_del_agua_del_Rio_Cauca_20260309.xlsx
    └── cruzar_calidad_caudal.py   ← este script

Uso:
    python cruzar_calidad_caudal.py

Requiere:
    pip install pandas openpyxl
"""

from pathlib import Path
import pandas as pd
import numpy as np
from openpyxl import load_workbook

# ── Configuración ──────────────────────────────────────────────────────────────
CALIDAD_FILE = Path("Calidad_del_agua_del_Rio_Cauca_20260501.xlsx")
EXCEL_NAME   = "caudal_2010.xlsx"
OUTPUT_FILE  = Path("calidad_clasificada.xlsx")
AÑO_DESDE    = 2010

MESES = ["ENE","FEB","MAR","ABR","MAY","JUN",
         "JUL","AGO","SEP","OCT","NOV","DIC"]

# ── Rangos de caudal por estación (PMC - Univ. del Valle; CVC 2001) ────────────
# Verano: caudal <= umbral_verano
# Transición: umbral_verano < caudal < umbral_invierno
# Invierno: caudal >= umbral_invierno
RANGOS_PMC = {
    "ANTES SUAREZ":       (91,  141),
    "ANTES RIO OVEJAS":   (93,  145),
    "ANTES RIO TIMBA":    (125, 197),
    "PASO DE LA BALSA":   (129, 204),
    "PASO DE LA BOLSA":   (150, 232),
    "PUENTE HORMIGUERO":  (177, 283),
    "JUANCHITO":          (179, 292),
    "PASO DEL COMERCIO":  (181, 295),
    "PUERTO ISAACS":      (184, 302),
    "PASO DE LA TORRE":   (189, 312),
    "VIJES":              (195, 322),
    "YOTOCO":             (203, 339),
    "MEDIACANOA":         (206, 344),
    "RIOFRIO":            (218, 374),
    "PUENTE GUAYABAL":    (230, 404),
    "LA VICTORIA":        (234, 411),
    "ANACARO":            (240, 428),
    "PUENTE LA VIRGINIA": (334, 556),
}

# Normalización de nombres con espacios dobles o variantes
NOMBRES_ALT = {
    "PUENTE  HORMIGUERO":    "PUENTE HORMIGUERO",
    "PASO DE  LA BALSA":     "PASO DE LA BALSA",
    "PASO DE  LA BOLSA":     "PASO DE LA BOLSA",
    "PASO DE  LA TORRE":     "PASO DE LA TORRE",
    "ANTES INTERCEPTOR":     "ANTES INTERCEPTOR",
    "ANTES INTERCEPTOR SUR": "ANTES INTERCEPTOR SUR",
}

# Estación de calidad → nombre de carpeta del proyecto
ESTACION_A_CARPETA = {
    "ANACARO":            "ANACARO",
    "LA VICTORIA":        "LA VICTORIA",
    "PUENTE HORMIGUERO":  "HORMIGUERO",
    "PUENTE GUAYABAL":    "GUAYABAL",
    "PASO DE LA BALSA":   "LA BALSA",
    "PASO DE LA BOLSA":   "LA BOLSA",
    "JUANCHITO":          "JUANCHITO (S HASTA 2017)",
    "MEDIACANOA":         "MEDICANOA",
}
# ───────────────────────────────────────────────────────────────────────────────


def normalizar_estacion(nombre: str) -> str:
    nombre = nombre.strip().upper()
    return NOMBRES_ALT.get(nombre, nombre)


def clasificar(caudal, estacion: str) -> str:
    """Clasifica la condición hidrológica según rangos PMC."""
    try:
        caudal = float(caudal)
    except (TypeError, ValueError):
        return "Sin caudal"
    if pd.isna(caudal):
        return "Sin caudal"
    rangos = RANGOS_PMC.get(estacion)
    if rangos is None:
        return "Sin rango PMC"
    v_max, i_min = rangos
    if caudal <= v_max:
        return "Verano"
    elif caudal >= i_min:
        return "Invierno"
    else:
        return "Transición"


def leer_caudal_diario(carpeta: Path) -> pd.DataFrame:
    """
    Lee el Excel de caudal y devuelve un DataFrame con columnas:
        FECHA (datetime), CAUDAL_DIARIO (float)
    """
    excel = carpeta / EXCEL_NAME
    if not excel.exists():
        return pd.DataFrame()

    wb = load_workbook(excel, data_only=True)
    registros = []

    for sheet_name in wb.sheetnames:
        try:
            año = int(sheet_name)
        except ValueError:
            continue

        ws = wb[sheet_name]
        for row in ws.iter_rows(min_row=3, values_only=True):
            if row[0] is None:
                continue
            try:
                dia = int(row[0])
            except (TypeError, ValueError):
                continue
            if not 1 <= dia <= 31:
                continue
            for mes_idx, val in enumerate(row[1:13]):
                if isinstance(val, (int, float)) and not pd.isna(val):
                    try:
                        fecha = pd.Timestamp(year=año, month=mes_idx+1, day=dia)
                        registros.append({"FECHA": fecha, "CAUDAL_DIARIO": float(val)})
                    except ValueError:
                        pass

    return pd.DataFrame(registros)


def main():
    # ── 1. Cargar calidad ──────────────────────────────────────────────────────
    if not CALIDAD_FILE.exists():
        raise FileNotFoundError(f"No se encontró '{CALIDAD_FILE}'")

    print(f"📄 Leyendo {CALIDAD_FILE} ...")
    df = pd.read_excel(CALIDAD_FILE)
    df['FECHA DE MUESTREO'] = pd.to_datetime(df['FECHA DE MUESTREO'], errors='coerce')
    df['ESTACION_STD'] = df['ESTACIONES'].apply(normalizar_estacion)

    # Filtrar fechas válidas y rango de años
    df = df[df['FECHA DE MUESTREO'].dt.year >= AÑO_DESDE].copy()
    df = df[df['FECHA DE MUESTREO'].notna()].copy()
    print(f"   → {len(df)} registros desde {AÑO_DESDE}")

    # Limpiar columna CAUDAL (m3/s): convertir a numérico y descartar > 5000
    df['CAUDAL_PROPIO'] = pd.to_numeric(df['CAUDAL (m3/s)'], errors='coerce')
    df.loc[df['CAUDAL_PROPIO'] > 5000, 'CAUDAL_PROPIO'] = np.nan

    # ── 2. Cargar caudales de carpetas del proyecto ────────────────────────────
    raiz = Path(".")
    carpetas = [d for d in raiz.iterdir() if d.is_dir() and (d / EXCEL_NAME).exists()]

    caudales_por_estacion = {}
    print(f"\n📂 Cargando caudales de {len(carpetas)} carpetas ...")

    for carpeta in sorted(carpetas):
        df_caudal = leer_caudal_diario(carpeta)
        if df_caudal.empty:
            continue
        nombre_carpeta = carpeta.name.upper()
        for est_std, carpeta_ref in ESTACION_A_CARPETA.items():
            if carpeta_ref.upper() == nombre_carpeta:
                caudales_por_estacion[est_std] = df_caudal
                print(f"   ✔  {carpeta.name:22s} → '{est_std}'")
        caudales_por_estacion[nombre_carpeta] = df_caudal

    # ── 3. Cruzar calidad con caudal por fecha ─────────────────────────────────
    print(f"\n🔗 Cruzando {len(df)} registros por fecha ...")

    caudales_cruzados = []
    fuente_caudal     = []

    for _, row in df.iterrows():
        fecha   = row['FECHA DE MUESTREO']
        est_std = row['ESTACION_STD']
        caudal  = np.nan
        fuente  = "Sin dato"

        # Buscar en Excel de caudal diario
        df_caud = caudales_por_estacion.get(est_std)
        if df_caud is not None and not df_caud.empty:
            match = df_caud[df_caud['FECHA'] == fecha]
            if not match.empty:
                caudal = match.iloc[0]['CAUDAL_DIARIO']
                fuente = "Excel caudal"

        # Fallback: usar caudal propio del Excel de calidad
        if pd.isna(caudal) and not pd.isna(row['CAUDAL_PROPIO']):
            caudal = row['CAUDAL_PROPIO']
            fuente = "Columna calidad"

        caudales_cruzados.append(caudal)
        fuente_caudal.append(fuente)

    df['CAUDAL_CRUZADO']  = caudales_cruzados
    df['FUENTE_CAUDAL']   = fuente_caudal

    # ── 4. Clasificar condición hidrológica ────────────────────────────────────
    df['CONDICION'] = df.apply(
        lambda r: clasificar(r['CAUDAL_CRUZADO'], r['ESTACION_STD']), axis=1
    )

    # ── 5. Resumen en consola ──────────────────────────────────────────────────
    print(f"\n📊 Clasificación general:")
    print(df['CONDICION'].value_counts().to_string())

    print(f"\n📊 Por estación:")
    resumen = (df.groupby(['ESTACION_STD','CONDICION'])
                 .size()
                 .unstack(fill_value=0)
                 .reindex(columns=['Verano','Transición','Invierno','Sin caudal','Sin rango PMC'],
                          fill_value=0))
    print(resumen.to_string())

    print(f"\n📊 Fuente del caudal usado:")
    print(df['FUENTE_CAUDAL'].value_counts().to_string())

    # ── 6. Guardar Excel resultado ─────────────────────────────────────────────
    cols_fijas = ['FECHA DE MUESTREO', 'ESTACIONES', 'ESTACION_STD',
                  'CAUDAL_CRUZADO', 'FUENTE_CAUDAL', 'CONDICION']
    cols_params = [c for c in df.columns
                   if c not in cols_fijas + ['CAUDAL (m3/s)', 'CAUDAL_PROPIO']]
    cols_orden = cols_fijas + cols_params + ['CAUDAL (m3/s)']
    cols_orden = [c for c in cols_orden if c in df.columns]

    df_out = df[cols_orden].sort_values(['ESTACION_STD', 'FECHA DE MUESTREO'])
    df_out.to_excel(OUTPUT_FILE, index=False)

    print(f"\n✅ Guardado: {OUTPUT_FILE.resolve()}")
    print(f"   {len(df_out)} registros | columnas nuevas: CAUDAL_CRUZADO, FUENTE_CAUDAL, CONDICION")


if __name__ == "__main__":
    main()

📄 Leyendo Calidad_del_agua_del_Rio_Cauca_20260501.xlsx ...
   → 930 registros desde 2010

📂 Cargando caudales de 18 carpetas ...
   ✔  ANACARO                → 'ANACARO'
   ✔  HORMIGUERO             → 'PUENTE HORMIGUERO'
   ✔  JUANCHITO (S HASTA 2017) → 'JUANCHITO'
   ✔  LA BALSA               → 'PASO DE LA BALSA'
   ✔  LA BOLSA               → 'PASO DE LA BOLSA'
   ✔  LA VICTORIA            → 'LA VICTORIA'
   ✔  MEDICANOA              → 'MEDIACANOA'

🔗 Cruzando 930 registros por fecha ...

📊 Clasificación general:
CONDICION
Sin caudal    683
Invierno       86
Transición     84
Verano         77

📊 Por estación:
CONDICION              Verano  Transición  Invierno  Sin caudal  Sin rango PMC
ESTACION_STD                                                                  
ANACARO                    15          16        15           3              0
ANTES INTERCEPTOR           0           0         0           6              0
ANTES INTERCEPTOR SUR       0           0         0          43 

In [16]:
"""
Genera la curva de duración de caudales para cada estación
y calcula los umbrales de Verano / Transición / Invierno
(percentiles 70% y 30% de permanencia).

Por cada estación genera:
    curva_duracion_caudales.png  — gráfica de la curva
    umbrales_caudal.txt          — resumen de umbrales calculados

Estructura esperada:
    <raíz>/
    ├── ANACARO/
    │   └── caudal_2010.xlsx
    ├── LA VICTORIA/
    │   └── caudal_2010.xlsx
    └── curva_duracion_caudales.py   ← este script

Uso:
    python curva_duracion_caudales.py

Requiere:
    pip install openpyxl matplotlib numpy
"""

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from openpyxl import load_workbook

# ── Configuración ──────────────────────────────────────────────
EXCEL_NAME    = "caudal_2010.xlsx"
DPI           = 150
PERM_VERANO   = 70   # % → caudales por debajo de este umbral = verano
PERM_INVIERNO = 30   # % → caudales por encima de este umbral = invierno
# ───────────────────────────────────────────────────────────────


def leer_caudales(excel_path: Path) -> np.ndarray:
    """Lee todos los valores diarios del Excel y devuelve un array 1D."""
    wb = load_workbook(excel_path, data_only=True)
    valores = []

    for sheet_name in wb.sheetnames:
        try:
            int(sheet_name)
        except ValueError:
            continue

        ws = wb[sheet_name]
        for row in ws.iter_rows(min_row=3, values_only=True):
            if row[0] is None:
                continue
            try:
                dia = int(row[0])
            except (TypeError, ValueError):
                continue
            if not 1 <= dia <= 31:
                continue
            # Ignorar filas de resumen (MÁXIMO, MÍNIMO, MEDIO)
            for val in row[1:13]:
                if isinstance(val, (int, float)) and not np.isnan(val):
                    valores.append(float(val))

    return np.array(valores)


def calcular_curva(caudales: np.ndarray):
    """
    Ordena los caudales de mayor a menor y calcula el porcentaje
    de permanencia (% del tiempo que se supera cada valor).
    Devuelve (permanencia_%, caudales_ordenados).
    """
    datos = np.sort(caudales)[::-1]              # mayor a menor
    n = len(datos)
    permanencia = np.arange(1, n + 1) / n * 100  # 0–100%
    return permanencia, datos


def calcular_umbrales(permanencia, caudales):
    """
    Devuelve el caudal correspondiente al percentil de permanencia
    solicitado (interpolando si es necesario).
    """
    umbral_verano   = float(np.interp(PERM_VERANO,   permanencia, caudales))
    umbral_invierno = float(np.interp(PERM_INVIERNO, permanencia, caudales))
    return umbral_verano, umbral_invierno


def graficar(estacion: str, permanencia, caudales,
             umbral_verano, umbral_invierno, out_path: Path):
    """Genera y guarda la gráfica de la curva de duración."""

    fig, ax = plt.subplots(figsize=(12, 6))
    fig.patch.set_facecolor("#F7F9FC")
    ax.set_facecolor("#F7F9FC")

    # ── Áreas de color por condición ──────────────────────────
    ax.axvspan(0,              PERM_INVIERNO, alpha=0.08, color="#2563EB",
               label="_nolegend_")
    ax.axvspan(PERM_INVIERNO,  PERM_VERANO,  alpha=0.08, color="#16A34A",
               label="_nolegend_")
    ax.axvspan(PERM_VERANO,    100,           alpha=0.08, color="#F59E0B",
               label="_nolegend_")

    # ── Curva principal ────────────────────────────────────────
    ax.plot(permanencia, caudales, color="#1E3A5F", linewidth=1.8,
            zorder=4, label="Caudal diario")

    # ── Líneas de umbral ───────────────────────────────────────
    ax.axvline(PERM_INVIERNO, color="#2563EB", linewidth=1.4,
               linestyle="--", zorder=5)
    ax.axvline(PERM_VERANO,   color="#F59E0B", linewidth=1.4,
               linestyle="--", zorder=5)

    ax.axhline(umbral_invierno, color="#2563EB", linewidth=1.0,
               linestyle=":", alpha=0.7, zorder=5)
    ax.axhline(umbral_verano,   color="#F59E0B", linewidth=1.0,
               linestyle=":", alpha=0.7, zorder=5)

    # ── Etiquetas de umbral ────────────────────────────────────
    ax.annotate(
        f"Invierno ≥ {umbral_invierno:,.0f} m³/s\n(permanencia < {PERM_INVIERNO}%)",
        xy=(PERM_INVIERNO, umbral_invierno),
        xytext=(PERM_INVIERNO + 2, umbral_invierno * 1.06),
        fontsize=8.5, color="#2563EB", fontweight="bold",
        arrowprops=dict(arrowstyle="-", color="#2563EB", lw=0.8)
    )
    ax.annotate(
        f"Verano ≤ {umbral_verano:,.0f} m³/s\n(permanencia > {PERM_VERANO}%)",
        xy=(PERM_VERANO, umbral_verano),
        xytext=(PERM_VERANO + 2, umbral_verano * 1.06),
        fontsize=8.5, color="#B45309", fontweight="bold",
        arrowprops=dict(arrowstyle="-", color="#B45309", lw=0.8)
    )

    # ── Etiquetas de zona ──────────────────────────────────────
    y_max = caudales.max()
    ax.text(PERM_INVIERNO / 2, y_max * 0.92, "INVIERNO",
            ha="center", fontsize=9, color="#2563EB",
            fontweight="bold", alpha=0.7)
    ax.text((PERM_INVIERNO + PERM_VERANO) / 2, y_max * 0.92, "TRANSICIÓN",
            ha="center", fontsize=9, color="#16A34A",
            fontweight="bold", alpha=0.7)
    ax.text((PERM_VERANO + 100) / 2, y_max * 0.92, "VERANO",
            ha="center", fontsize=9, color="#B45309",
            fontweight="bold", alpha=0.7)

    # ── Formato de ejes ────────────────────────────────────────
    ax.set_xlim(0, 100)
    ax.set_ylim(0, y_max * 1.05)
    ax.set_xlabel("Porcentaje de permanencia (%)", fontsize=10, labelpad=8)
    ax.set_ylabel("Caudal (m³/s)", fontsize=10, labelpad=8)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.xaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
    ax.grid(axis="both", linestyle="--", alpha=0.35, zorder=0)

    ax.set_title(
        f"Curva de Duración de Caudales — Estación {estacion}\n"
        f"Datos diarios 2010–2026  |  n = {len(caudales):,} días",
        fontsize=12, fontweight="bold", pad=14, color="#1A1A2E"
    )

    fig.tight_layout()
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close(fig)


def guardar_umbrales(estacion: str, caudales: np.ndarray,
                     umbral_verano, umbral_invierno, out_path: Path):
    """Guarda un resumen de texto con los umbrales calculados."""
    lineas = [
        f"Estación: {estacion}",
        f"Datos: {len(caudales):,} caudales diarios (2010–2026)",
        f"",
        f"── Umbrales calculados ──────────────────────────────",
        f"Permanencia usada: {PERM_INVIERNO}% (invierno) | {PERM_VERANO}% (verano)",
        f"",
        f"  INVIERNO  : caudal ≥ {umbral_invierno:,.1f} m³/s  (permanencia < {PERM_INVIERNO}%)",
        f"  TRANSICIÓN: {umbral_verano:,.1f} – {umbral_invierno:,.1f} m³/s",
        f"  VERANO    : caudal ≤ {umbral_verano:,.1f} m³/s  (permanencia > {PERM_VERANO}%)",
        f"",
        f"── Estadísticas generales ───────────────────────────",
        f"  Mínimo   : {caudales.min():,.1f} m³/s",
        f"  Máximo   : {caudales.max():,.1f} m³/s",
        f"  Promedio : {caudales.mean():,.1f} m³/s",
        f"  Mediana  : {np.median(caudales):,.1f} m³/s",
    ]
    out_path.write_text("\n".join(lineas), encoding="utf-8")


def procesar_estacion(carpeta: Path):
    excel = carpeta / EXCEL_NAME
    estacion = carpeta.name.upper()

    caudales = leer_caudales(excel)
    if len(caudales) == 0:
        print(f"  ⚠  {estacion:22s} — sin datos")
        return

    permanencia, caudales_ord = calcular_curva(caudales)
    umbral_verano, umbral_invierno = calcular_umbrales(permanencia, caudales_ord)

    graficar(
        estacion, permanencia, caudales_ord,
        umbral_verano, umbral_invierno,
        carpeta / "curva_duracion_caudales.png"
    )
    guardar_umbrales(
        estacion, caudales,
        umbral_verano, umbral_invierno,
        carpeta / "umbrales_caudal.txt"
    )

    print(f"  ✔  {estacion:22s} "
          f"| n={len(caudales):,} "
          f"| Verano ≤ {umbral_verano:,.0f}  "
          f"| Invierno ≥ {umbral_invierno:,.0f}  m³/s")


def main():
    raiz = Path(".")
    estaciones = sorted([
        d for d in raiz.iterdir()
        if d.is_dir() and (d / EXCEL_NAME).exists()
    ])

    if not estaciones:
        print("❌ No se encontraron carpetas con 'caudal_2010.xlsx'.")
        print("   Corre el script desde la raíz del proyecto.")
        return

    print(f"🗂  {len(estaciones)} estaciones encontradas\n")
    print(f"  {'Estación':22s} | {'Datos':>8} | {'Verano':>12} | {'Invierno':>12}")
    print(f"  {'-'*22} | {'-'*8} | {'-'*12} | {'-'*12}")

    for carpeta in estaciones:
        procesar_estacion(carpeta)

    print(f"\n✅ Listo. Archivos generados en cada carpeta:")
    print(f"   • curva_duracion_caudales.png")
    print(f"   • umbrales_caudal.txt")


if __name__ == "__main__":
    main()

🗂  18 estaciones encontradas

  Estación               |    Datos |       Verano |     Invierno
  ---------------------- | -------- | ------------ | ------------
  ✔  ANACARO                | n=5,868 | Verano ≤ 232  | Invierno ≥ 494  m³/s
  ✔  GARZONERO NORTE        | n=1,428 | Verano ≤ 1  | Invierno ≥ 3  m³/s
  ✔  GARZONERO SUR (S - 2023) | n=3,000 | Verano ≤ 2  | Invierno ≥ 4  m³/s
  ✔  GUAYABAL (HASTA 2022)  | n=4,658 | Verano ≤ 245  | Invierno ≥ 482  m³/s
  ✔  HORMIGUERO             | n=1,050 | Verano ≤ 214  | Invierno ≥ 330  m³/s
  ✔  JUAN DIAZ              | n=2,043 | Verano ≤ 2  | Invierno ≥ 4  m³/s
  ✔  JUANCHITO (S HASTA 2017) | n=1,094 | Verano ≤ 151  | Invierno ≥ 256  m³/s
  ✔  LA BALSA               | n=5,564 | Verano ≤ 124  | Invierno ≥ 219  m³/s
  ✔  LA BOLSA               | n=446 | Verano ≤ 137  | Invierno ≥ 200  m³/s
  ⚠  LA FLORESTA            — sin datos
  ⚠  LA PRIMAVERA           — sin datos
  ✔  LA VICTORIA            | n=5,883 | Verano ≤ 229  | Invierno ≥ 483  m³/

In [17]:
"""
Genera el perfil longitudinal de caudal del río Cauca
por período de 4 años y condición hidrológica (Invierno/Transición/Verano),
al estilo de la Figura 5.10 del informe PMC (CVC - Univ. del Valle).

Períodos de análisis (4 años cada uno, 2015–2026):
    2015–2018 | 2019–2022 | 2023–2026

Abscisados desde La Balsa (km 0):
    Fuente: Figura 5.11, Informe PMC (Univ. del Valle - CVC)
    Solo se incluyen estaciones con abscisado confirmado en el PMC.
    Tablanca, La Primavera y La Floresta excluidas por falta de valor
    de referencia — agregar cuando se tenga esquematización PMC o SIC.

Estructura esperada:
    <raíz>/
    ├── LA BALSA/
    │   └── caudal_2010.xlsx
    ├── HORMIGUERO/
    │   └── caudal_2010.xlsx
    ├── MEDICANOA/
    │   └── caudal_2010.xlsx
    ├── GUAYABAL/
    │   └── caudal_2010.xlsx
    ├── LA VICTORIA/
    │   └── caudal_2010.xlsx
    ├── ANACARO/
    │   └── caudal_2010.xlsx
    └── perfil_longitudinal_caudal.py   ← este script

Salida:
    perfil_longitudinal_caudal.png  (en la raíz del proyecto)

Uso:
    python perfil_longitudinal_caudal.py

Requiere:
    pip install openpyxl matplotlib numpy pandas
"""

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.lines as mlines
from openpyxl import load_workbook

# ── Configuración ──────────────────────────────────────────────────────────────
EXCEL_NAME    = "caudal_2010.xlsx"
OUTPUT_FILE   = Path("perfil_longitudinal_caudal.png")
DPI           = 180
PERM_VERANO   = 70   # % permanencia → umbral verano
PERM_INVIERNO = 30   # % permanencia → umbral invierno

# Períodos de 4 años — 2015 a 2026
PERIODOS = {
    "2015–2018": (2015, 2018),
    "2019–2022": (2019, 2022),
    "2023–2026": (2023, 2026),
}

ESTILOS_LINEA = {
    "2015–2018": "-",
    "2019–2022": "--",
    "2023–2026": ":",
}

ESTILOS_COND = {
    "Invierno":   {"color": "#1565C0", "marker": "o"},
    "Transición": {"color": "#2E7D32", "marker": "^"},
    "Verano":     {"color": "#B71C1C", "marker": "s"},
}

# ── Abscisados confirmados desde La Balsa (km 0) ──────────────────────────────
# Fuente: Figura 5.11, Informe PMC (Univ. del Valle - CVC)
# Valores originales PMC menos 75 km (posición de La Balsa en el PMC)
# ⚠ Tablanca, La Primavera y La Floresta excluidas — sin valor PMC confirmado
# ⚠ Pendiente verificación con esquematización del informe de modelación PMC
ESTACIONES_INFO = {
    "LA BALSA":     {"abscisado": 0},
    "TABLANCA":     {"abscisado": -15},
    "HORMIGUERO":   {"abscisado": 50},
    "MEDICANOA":    {"abscisado": 145},
    "GUAYABAL":     {"abscisado": 270},
    "LA VICTORIA":  {"abscisado": 290},
    "ANACARO":      {"abscisado": 340},
}

NOMBRES_MAPA = {
    "LA BALSA":     "La Balsa",
    "TABLANCA":     "Tablanca",
    "HORMIGUERO":   "Hormiguero",
    "MEDICANOA":    "Mediacanoa",
    "GUAYABAL":     "Guayabal",
    "LA VICTORIA":  "La Victoria",
    "ANACARO":      "Anacaro",
}
# ───────────────────────────────────────────────────────────────────────────────


def periodo_label(año):
    for label, (ini, fin) in PERIODOS.items():
        if ini <= año <= fin:
            return label
    return None


def leer_caudales(excel_path: Path) -> pd.DataFrame:
    wb = load_workbook(excel_path, data_only=True)
    registros = []
    for sheet_name in wb.sheetnames:
        try:
            año = int(sheet_name)
        except ValueError:
            continue
        ws = wb[sheet_name]
        for row in ws.iter_rows(min_row=3, values_only=True):
            if row[0] is None:
                continue
            try:
                dia = int(row[0])
            except (TypeError, ValueError):
                continue
            if not 1 <= dia <= 31:
                continue
            for mes_idx, val in enumerate(row[1:13]):
                if isinstance(val, (int, float)) and not np.isnan(val):
                    try:
                        fecha = pd.Timestamp(year=año, month=mes_idx+1, day=dia)
                        registros.append({"FECHA": fecha, "CAUDAL": float(val)})
                    except ValueError:
                        pass
    return pd.DataFrame(registros)


def calcular_umbrales(caudales: np.ndarray):
    """Umbrales calculados con toda la serie histórica disponible."""
    datos = np.sort(caudales)[::-1]
    permanencia = np.arange(1, len(datos) + 1) / len(datos) * 100
    u_v = float(np.interp(PERM_VERANO,   permanencia, datos))
    u_i = float(np.interp(PERM_INVIERNO, permanencia, datos))
    return u_v, u_i


def clasificar_serie(df: pd.DataFrame, u_v, u_i) -> pd.DataFrame:
    def cond(q):
        if q <= u_v:   return "Verano"
        elif q >= u_i: return "Invierno"
        else:          return "Transición"

    df = df.copy()
    df["CONDICION"] = df["CAUDAL"].apply(cond)
    df["PERIODO"]   = df["FECHA"].dt.year.apply(periodo_label)
    return df[df["PERIODO"].notna()]


def main():
    raiz = Path(".")
    carpetas = sorted([
        d for d in raiz.iterdir()
        if d.is_dir() and (d / EXCEL_NAME).exists()
    ])

    if not carpetas:
        print("❌ No se encontraron carpetas con 'caudal_2010.xlsx'.")
        return

    print(f"🗂  {len(carpetas)} carpetas encontradas\n")
    print(f"  {'Estación':22s} | {'Absc.(km)':>10} | {'Verano':>10} | {'Invierno':>10}")
    print(f"  {'-'*22} | {'-'*10} | {'-'*10} | {'-'*10}")

    datos_grafica = []

    for carpeta in carpetas:
        nombre = carpeta.name.upper()
        info   = ESTACIONES_INFO.get(nombre)

        if info is None:
            print(f"  ✗  {nombre:22s} — excluida (sin abscisado PMC confirmado)")
            continue

        df = leer_caudales(carpeta / EXCEL_NAME)
        if df.empty:
            print(f"  ⚠  {nombre:22s} — sin datos de caudal")
            continue

        u_v, u_i = calcular_umbrales(df["CAUDAL"].values)
        df_c = clasificar_serie(df, u_v, u_i)

        resumen = (df_c.groupby(["PERIODO", "CONDICION"])["CAUDAL"]
                       .mean().round(1))

        for (periodo, cond), prom in resumen.items():
            datos_grafica.append({
                "ESTACION":  nombre,
                "ABSCISADO": info["abscisado"],
                "PERIODO":   periodo,
                "CONDICION": cond,
                "PROMEDIO":  prom,
            })

        print(f"  ✔  {nombre:22s} | {info['abscisado']:>9.0f} "
              f"| {u_v:>9.0f}  | {u_i:>9.0f}  m³/s")

    if not datos_grafica:
        print("\n❌ Sin datos suficientes para graficar.")
        return

    df_g = pd.DataFrame(datos_grafica).sort_values("ABSCISADO")

    # ── Graficar ───────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(14, 7))
    fig.patch.set_facecolor("#F7F9FC")
    ax.set_facecolor("#F7F9FC")

    for cond, estilo_c in ESTILOS_COND.items():
        for periodo, lstyle in ESTILOS_LINEA.items():
            sub = (df_g[(df_g["CONDICION"] == cond) &
                        (df_g["PERIODO"]   == periodo)]
                   .sort_values("ABSCISADO"))
            if sub.empty:
                continue
            ax.plot(
                sub["ABSCISADO"], sub["PROMEDIO"],
                color=estilo_c["color"], linestyle=lstyle,
                linewidth=1.8, marker=estilo_c["marker"],
                markersize=6, markerfacecolor="white",
                markeredgecolor=estilo_c["color"], markeredgewidth=2,
                zorder=4, alpha=0.9
            )

    # ── Etiquetas anti-superposición ───────────────────────────────────────────
    MIN_SEP_PX = 18
    puntos_labels = []
    for cond, estilo_c in ESTILOS_COND.items():
        for periodo in PERIODOS:
            sub = (df_g[(df_g["CONDICION"] == cond) &
                        (df_g["PERIODO"]   == periodo)]
                   .sort_values("ABSCISADO"))
            for _, row in sub.iterrows():
                puntos_labels.append({
                    "x":     row["ABSCISADO"],
                    "y":     row["PROMEDIO"],
                    "texto": f"{row['PROMEDIO']:,.0f}",
                    "color": estilo_c["color"],
                })

    df_lbl = pd.DataFrame(puntos_labels).sort_values(["x", "y"])
    for x_val, grp in df_lbl.groupby("x"):
        grp = grp.sort_values("y").reset_index(drop=True)
        offsets_calc = [10] * len(grp)
        for k in range(1, len(grp)):
            diff = grp.loc[k, "y"] - grp.loc[k-1, "y"]
            if diff < 25:
                offsets_calc[k] = offsets_calc[k-1] + MIN_SEP_PX
        for k, (_, lrow) in enumerate(grp.iterrows()):
            ax.annotate(
                lrow["texto"],
                xy=(lrow["x"], lrow["y"]),
                xytext=(0, offsets_calc[k]),
                textcoords="offset points",
                ha="center", fontsize=6.5,
                color=lrow["color"], alpha=0.9,
                fontweight="bold",
            )

    # ── Eje X: estaciones con km ───────────────────────────────────────────────
    estaciones_presentes = (df_g[["ESTACION", "ABSCISADO"]]
                             .drop_duplicates()
                             .sort_values("ABSCISADO"))

    for _, row in estaciones_presentes.iterrows():
        ax.axvline(row["ABSCISADO"], color="#CCCCCC",
                   linewidth=0.6, linestyle="-", zorder=1)

    ax.set_xticks(estaciones_presentes["ABSCISADO"])
    labels_km = [
        f"{NOMBRES_MAPA.get(n, n.title())}\n({a:.0f} km)"
        for n, a in zip(estaciones_presentes["ESTACION"],
                        estaciones_presentes["ABSCISADO"])
    ]
    ax.set_xticklabels(labels_km, rotation=0, ha="center", fontsize=9)

    # ── Leyenda ────────────────────────────────────────────────────────────────
    legend_cond = [
        mlines.Line2D([], [], color=v["color"], marker=v["marker"],
                      markersize=6, markerfacecolor="white",
                      markeredgecolor=v["color"], linewidth=0,
                      label=cond)
        for cond, v in ESTILOS_COND.items()
    ]
    legend_periodo = [
        mlines.Line2D([], [], color="#555555", linestyle=ls,
                      linewidth=1.8, label=p)
        for p, ls in ESTILOS_LINEA.items()
    ]
    leg1 = ax.legend(handles=legend_cond, title="Condición",
                     loc="upper left", fontsize=9, title_fontsize=9,
                     framealpha=0.85)
    ax.legend(handles=legend_periodo, title="Período",
              loc="upper left", bbox_to_anchor=(0.13, 1.0),
              fontsize=9, title_fontsize=9, framealpha=0.85)
    ax.add_artist(leg1)

    # ── Formato general ────────────────────────────────────────────────────────
    x_min = estaciones_presentes["ABSCISADO"].min()
    x_max = estaciones_presentes["ABSCISADO"].max()
    y_max = df_g["PROMEDIO"].max()
    ax.set_xlim(x_min - 15, x_max + 20)
    ax.set_ylim(0, y_max * 1.20)
    ax.set_xlabel(
        "Abscisado (km desde La Balsa)  —  "
        "Fuente: PMC Fig. 5.11 (Univ. del Valle - CVC)  ",
        fontsize=9, labelpad=8, color="#555555"
    )
    ax.set_ylabel("Caudal promedio (m³/s)", fontsize=10, labelpad=8)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.grid(axis="y", linestyle="--", alpha=0.35, zorder=0)

    ax.set_title(
        "Variación del Caudal en el Río Cauca — 2015–2026\n"
        "Períodos 2015–2018 | 2019–2022 | 2023–2026  |  "
        "Condiciones: Invierno – Transición – Verano",
        fontsize=12, fontweight="bold", pad=14, color="#1A1A2E"
    )

    fig.text(0.99, 0.01,
             "Umbrales calculados con curva de duración de caudales "
             "(percentil 30%/70%) por estación  |  Fuente: CVC 2015–2026",
             ha="right", fontsize=6.5, color="#888888", style="italic")

    fig.tight_layout()
    fig.savefig(OUTPUT_FILE, dpi=DPI, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"\n✅ Gráfica guardada: {OUTPUT_FILE.resolve()}")


if __name__ == "__main__":
    main()

🗂  18 carpetas encontradas

  Estación               |  Absc.(km) |     Verano |   Invierno
  ---------------------- | ---------- | ---------- | ----------
  ✔  ANACARO                |       340 |       232  |       494  m³/s
  ✗  GARZONERO NORTE        — excluida (sin abscisado PMC confirmado)
  ✗  GARZONERO SUR (S - 2023) — excluida (sin abscisado PMC confirmado)
  ✗  GUAYABAL (HASTA 2022)  — excluida (sin abscisado PMC confirmado)
  ✔  HORMIGUERO             |        50 |       213  |       330  m³/s
  ✗  JUAN DIAZ              — excluida (sin abscisado PMC confirmado)
  ✗  JUANCHITO (S HASTA 2017) — excluida (sin abscisado PMC confirmado)
  ✔  LA BALSA               |         0 |       124  |       218  m³/s
  ✗  LA BOLSA               — excluida (sin abscisado PMC confirmado)
  ✗  LA FLORESTA            — excluida (sin abscisado PMC confirmado)
  ✗  LA PRIMAVERA           — excluida (sin abscisado PMC confirmado)
  ✔  LA VICTORIA            |       290 |       228  |       482  m

In [18]:
"""
Genera el perfil longitudinal de caudal del río Cauca
por período de 4 años y condición hidrológica (Invierno/Transición/Verano),
al estilo de la Figura 5.10 del informe PMC (CVC - Univ. del Valle).

Períodos de análisis (4 años cada uno, 2015–2026):
    2015–2018 | 2019–2022 | 2023–2026

Abscisados desde La Balsa (km 0):
    Fuente: Figura 5.11, Informe PMC (Univ. del Valle - CVC)
    Solo se incluyen estaciones con abscisado confirmado en el PMC.
    Tablanca, La Primavera y La Floresta excluidas por falta de valor
    de referencia — agregar cuando se tenga esquematización PMC o SIC.

Estructura esperada:
    <raíz>/
    ├── LA BALSA/
    │   └── caudal_2010.xlsx
    ├── HORMIGUERO/
    │   └── caudal_2010.xlsx
    ├── MEDICANOA/
    │   └── caudal_2010.xlsx
    ├── GUAYABAL/
    │   └── caudal_2010.xlsx
    ├── LA VICTORIA/
    │   └── caudal_2010.xlsx
    ├── ANACARO/
    │   └── caudal_2010.xlsx
    └── perfil_longitudinal_caudal.py   ← este script

Salida:
    perfil_longitudinal_caudal.png  (en la raíz del proyecto)

Uso:
    python perfil_longitudinal_caudal.py

Requiere:
    pip install openpyxl matplotlib numpy pandas
"""

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.lines as mlines
from openpyxl import load_workbook

# ── Configuración ──────────────────────────────────────────────────────────────
EXCEL_NAME    = "caudal_2010.xlsx"
OUTPUT_FILE   = Path("perfil_longitudinal_caudal.png")
DPI           = 180
PERM_VERANO   = 70   # % permanencia → umbral verano
PERM_INVIERNO = 30   # % permanencia → umbral invierno

# Períodos de 4 años — 2015 a 2026
PERIODOS = {
    "2015–2018": (2015, 2018),
    "2019–2022": (2019, 2022),
    "2023–2026": (2023, 2026),
}

ESTILOS_LINEA = {
    "2015–2018": "-",
    "2019–2022": "--",
    "2023–2026": ":",
}

ESTILOS_COND = {
    "Invierno":   {"color": "#1565C0", "marker": "o"},
    "Transición": {"color": "#2E7D32", "marker": "^"},
    "Verano":     {"color": "#B71C1C", "marker": "s"},
}

# ── Abscisados confirmados desde La Balsa (km 0) ──────────────────────────────
# Fuente: Figura 5.11, Informe PMC (Univ. del Valle - CVC)
# Valores originales PMC menos 75 km (posición de La Balsa en el PMC)
# ⚠ Tablanca, La Primavera y La Floresta excluidas — sin valor PMC confirmado
# ⚠ Pendiente verificación con esquematización del informe de modelación PMC
ESTACIONES_INFO = {
    "LA BALSA":     {"abscisado": 0},
    "TABLANCA":     {"abscisado": -15},
    "HORMIGUERO":   {"abscisado": 50},
    "MEDICANOA":    {"abscisado": 145},
    "GUAYABAL":     {"abscisado": 270},
    "LA VICTORIA":  {"abscisado": 290},
    "ANACARO":      {"abscisado": 340},
}

NOMBRES_MAPA = {
    "LA BALSA":     "La Balsa",
    "TABLANCA":     "Tablanca",
    "HORMIGUERO":   "Hormiguero",
    "MEDICANOA":    "Mediacanoa",
    "GUAYABAL":     "Guayabal",
    "LA VICTORIA":  "La Victoria",
    "ANACARO":      "Anacaro",
}
# ───────────────────────────────────────────────────────────────────────────────


def periodo_label(año):
    for label, (ini, fin) in PERIODOS.items():
        if ini <= año <= fin:
            return label
    return None


def leer_caudales(excel_path: Path) -> pd.DataFrame:
    wb = load_workbook(excel_path, data_only=True)
    registros = []
    for sheet_name in wb.sheetnames:
        try:
            año = int(sheet_name)
        except ValueError:
            continue
        ws = wb[sheet_name]
        for row in ws.iter_rows(min_row=3, values_only=True):
            if row[0] is None:
                continue
            try:
                dia = int(row[0])
            except (TypeError, ValueError):
                continue
            if not 1 <= dia <= 31:
                continue
            for mes_idx, val in enumerate(row[1:13]):
                if isinstance(val, (int, float)) and not np.isnan(val):
                    try:
                        fecha = pd.Timestamp(year=año, month=mes_idx+1, day=dia)
                        registros.append({"FECHA": fecha, "CAUDAL": float(val)})
                    except ValueError:
                        pass
    return pd.DataFrame(registros)


def calcular_umbrales(caudales: np.ndarray):
    """Umbrales calculados con toda la serie histórica disponible."""
    datos = np.sort(caudales)[::-1]
    permanencia = np.arange(1, len(datos) + 1) / len(datos) * 100
    u_v = float(np.interp(PERM_VERANO,   permanencia, datos))
    u_i = float(np.interp(PERM_INVIERNO, permanencia, datos))
    return u_v, u_i


def clasificar_serie(df: pd.DataFrame, u_v, u_i) -> pd.DataFrame:
    def cond(q):
        if q <= u_v:   return "Verano"
        elif q >= u_i: return "Invierno"
        else:          return "Transición"

    df = df.copy()
    df["CONDICION"] = df["CAUDAL"].apply(cond)
    df["PERIODO"]   = df["FECHA"].dt.year.apply(periodo_label)
    return df[df["PERIODO"].notna()]


def main():
    raiz = Path(".")
    carpetas = sorted([
        d for d in raiz.iterdir()
        if d.is_dir() and (d / EXCEL_NAME).exists()
    ])

    if not carpetas:
        print("❌ No se encontraron carpetas con 'caudal_2010.xlsx'.")
        return

    print(f"🗂  {len(carpetas)} carpetas encontradas\n")
    print(f"  {'Estación':22s} | {'Absc.(km)':>10} | {'Verano':>10} | {'Invierno':>10}")
    print(f"  {'-'*22} | {'-'*10} | {'-'*10} | {'-'*10}")

    datos_grafica = []

    for carpeta in carpetas:
        nombre = carpeta.name.upper()
        info   = ESTACIONES_INFO.get(nombre)

        if info is None:
            print(f"  ✗  {nombre:22s} — excluida (sin abscisado PMC confirmado)")
            continue

        df = leer_caudales(carpeta / EXCEL_NAME)
        if df.empty:
            print(f"  ⚠  {nombre:22s} — sin datos de caudal")
            continue

        u_v, u_i = calcular_umbrales(df["CAUDAL"].values)
        df_c = clasificar_serie(df, u_v, u_i)

        resumen = (df_c.groupby(["PERIODO", "CONDICION"])["CAUDAL"]
                       .mean().round(1))

        for (periodo, cond), prom in resumen.items():
            datos_grafica.append({
                "ESTACION":  nombre,
                "ABSCISADO": info["abscisado"],
                "PERIODO":   periodo,
                "CONDICION": cond,
                "PROMEDIO":  prom,
            })

        print(f"  ✔  {nombre:22s} | {info['abscisado']:>9.0f} "
              f"| {u_v:>9.0f}  | {u_i:>9.0f}  m³/s")

    if not datos_grafica:
        print("\n❌ Sin datos suficientes para graficar.")
        return

    df_g = pd.DataFrame(datos_grafica).sort_values("ABSCISADO")

    # ── Graficar ───────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(14, 7))
    fig.patch.set_facecolor("#F7F9FC")
    ax.set_facecolor("#F7F9FC")

    for cond, estilo_c in ESTILOS_COND.items():
        for periodo, lstyle in ESTILOS_LINEA.items():
            sub = (df_g[(df_g["CONDICION"] == cond) &
                        (df_g["PERIODO"]   == periodo)]
                   .sort_values("ABSCISADO"))
            if sub.empty:
                continue
            ax.plot(
                sub["ABSCISADO"], sub["PROMEDIO"],
                color=estilo_c["color"], linestyle=lstyle,
                linewidth=1.8, marker=estilo_c["marker"],
                markersize=6, markerfacecolor="white",
                markeredgecolor=estilo_c["color"], markeredgewidth=2,
                zorder=4, alpha=0.9
            )

    # ── Etiquetas anti-superposición ───────────────────────────────────────────
    MIN_SEP_PX = 18
    puntos_labels = []
    for cond, estilo_c in ESTILOS_COND.items():
        for periodo in PERIODOS:
            sub = (df_g[(df_g["CONDICION"] == cond) &
                        (df_g["PERIODO"]   == periodo)]
                   .sort_values("ABSCISADO"))
            for _, row in sub.iterrows():
                puntos_labels.append({
                    "x":     row["ABSCISADO"],
                    "y":     row["PROMEDIO"],
                    "texto": f"{row['PROMEDIO']:,.0f}",
                    "color": estilo_c["color"],
                })

    df_lbl = pd.DataFrame(puntos_labels).sort_values(["x", "y"])
    for x_val, grp in df_lbl.groupby("x"):
        grp = grp.sort_values("y").reset_index(drop=True)
        offsets_calc = [10] * len(grp)
        for k in range(1, len(grp)):
            diff = grp.loc[k, "y"] - grp.loc[k-1, "y"]
            if diff < 25:
                offsets_calc[k] = offsets_calc[k-1] + MIN_SEP_PX
        for k, (_, lrow) in enumerate(grp.iterrows()):
            ax.annotate(
                lrow["texto"],
                xy=(lrow["x"], lrow["y"]),
                xytext=(0, offsets_calc[k]),
                textcoords="offset points",
                ha="center", fontsize=6.5,
                color=lrow["color"], alpha=0.9,
                fontweight="bold",
            )

    # ── Eje X: estaciones con km ───────────────────────────────────────────────
    estaciones_presentes = (df_g[["ESTACION", "ABSCISADO"]]
                             .drop_duplicates()
                             .sort_values("ABSCISADO"))

    for _, row in estaciones_presentes.iterrows():
        ax.axvline(row["ABSCISADO"], color="#CCCCCC",
                   linewidth=0.6, linestyle="-", zorder=1)

    ax.set_xticks(estaciones_presentes["ABSCISADO"])
    labels_km = [
        f"{NOMBRES_MAPA.get(n, n.title())}\n({a:.0f} km)"
        for n, a in zip(estaciones_presentes["ESTACION"],
                        estaciones_presentes["ABSCISADO"])
    ]
    ax.set_xticklabels(labels_km, rotation=0, ha="center", fontsize=9)

    # ── Leyenda ────────────────────────────────────────────────────────────────
    legend_cond = [
        mlines.Line2D([], [], color=v["color"], marker=v["marker"],
                      markersize=6, markerfacecolor="white",
                      markeredgecolor=v["color"], linewidth=0,
                      label=cond)
        for cond, v in ESTILOS_COND.items()
    ]
    legend_periodo = [
        mlines.Line2D([], [], color="#555555", linestyle=ls,
                      linewidth=1.8, label=p)
        for p, ls in ESTILOS_LINEA.items()
    ]
    leg1 = ax.legend(handles=legend_cond, title="Condición",
                     loc="upper left", fontsize=9, title_fontsize=9,
                     framealpha=0.85)
    ax.legend(handles=legend_periodo, title="Período",
              loc="upper left", bbox_to_anchor=(0.13, 1.0),
              fontsize=9, title_fontsize=9, framealpha=0.85)
    ax.add_artist(leg1)

    # ── Formato general ────────────────────────────────────────────────────────
    x_min = estaciones_presentes["ABSCISADO"].min()
    x_max = estaciones_presentes["ABSCISADO"].max()
    y_max = df_g["PROMEDIO"].max()
    ax.set_xlim(x_min - 15, x_max + 20)
    ax.set_ylim(0, y_max * 1.20)
    ax.set_xlabel(
        "Abscisado (km desde La Balsa)  —  "
        "Fuente: PMC Fig. 5.11 (Univ. del Valle - CVC)  ",
        fontsize=9, labelpad=8, color="#555555"
    )
    ax.set_ylabel("Caudal promedio (m³/s)", fontsize=10, labelpad=8)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.grid(axis="y", linestyle="--", alpha=0.35, zorder=0)

    ax.set_title(
        "Variación del Caudal en el Río Cauca — 2015–2026\n"
        "Períodos 2015–2018 | 2019–2022 | 2023–2026  |  "
        "Condiciones: Invierno – Transición – Verano",
        fontsize=12, fontweight="bold", pad=14, color="#1A1A2E"
    )

    fig.text(0.99, 0.01,
             "Umbrales calculados con curva de duración de caudales "
             "(percentil 30%/70%) por estación  |  Fuente: CVC 2015–2026",
             ha="right", fontsize=6.5, color="#888888", style="italic")

    fig.tight_layout()
    fig.savefig(OUTPUT_FILE, dpi=DPI, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"\n✅ Gráfica guardada: {OUTPUT_FILE.resolve()}")


if __name__ == "__main__":
    main()

🗂  18 carpetas encontradas

  Estación               |  Absc.(km) |     Verano |   Invierno
  ---------------------- | ---------- | ---------- | ----------
  ✔  ANACARO                |       340 |       232  |       494  m³/s
  ✗  GARZONERO NORTE        — excluida (sin abscisado PMC confirmado)
  ✗  GARZONERO SUR (S - 2023) — excluida (sin abscisado PMC confirmado)
  ✗  GUAYABAL (HASTA 2022)  — excluida (sin abscisado PMC confirmado)
  ✔  HORMIGUERO             |        50 |       213  |       330  m³/s
  ✗  JUAN DIAZ              — excluida (sin abscisado PMC confirmado)
  ✗  JUANCHITO (S HASTA 2017) — excluida (sin abscisado PMC confirmado)
  ✔  LA BALSA               |         0 |       124  |       218  m³/s
  ✗  LA BOLSA               — excluida (sin abscisado PMC confirmado)
  ✗  LA FLORESTA            — excluida (sin abscisado PMC confirmado)
  ✗  LA PRIMAVERA           — excluida (sin abscisado PMC confirmado)
  ✔  LA VICTORIA            |       290 |       228  |       482  m

In [20]:
"""
Genera perfiles longitudinales de calidad del agua del río Cauca
clasificados por condición hidrológica (Invierno / Transición / Verano),
usando Mediacanoa como estación de referencia para la clasificación.

Metodología:
    - Para cada campaña de monitoreo se busca el caudal promedio diario
      de Mediacanoa en esa fecha
    - Con la curva de duración de caudales de Mediacanoa (percentil 30%/70%)
      se clasifica la campaña completa como Invierno, Transición o Verano
    - Se grafican los promedios de cada parámetro por condición hidrológica

Por cada parámetro genera dos gráficas:
  1. perfil_<PARAM>_condicion.png  — promedio por condición hidrológica
  2. perfil_<PARAM>_boxplot.png    — distribución por condición (box plot)

Parámetros: DBO, DQO, OD, SST, Nitratos, Fósforo Total

Notas de calidad de datos:
    - NITRATOS: 57 registros de las campañas de julio y agosto 2022 tienen
      valores físicamente imposibles (hasta 150,000,000 mg/L vs máx normal
      de ~5 mg/L). Se filtran automáticamente con límite de 15 mg/L.
      Origen probable: error de unidades o digitación en la fuente CVC.
      Se recomienda reportar este error a la CVC para corrección.

Estructura esperada:
    <raíz>/
    ├── MEDICANOA/
    │   └── caudal_2010.xlsx          ← referencia hidrológica
    ├── Calidad_del_agua_del_Rio_Cauca_20260309.xlsx
    └── perfil_longitudinal_calidad.py

Uso:
    python perfil_longitudinal_calidad.py

Requiere:
    pip install pandas openpyxl matplotlib numpy
"""

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
from openpyxl import load_workbook

# ── Configuración ──────────────────────────────────────────────────────────────
CALIDAD_FILE      = Path("Calidad_del_agua_del_Rio_Cauca_20260501.xlsx")
CAUDAL_REF_FILE   = Path("MEDICANOA/caudal_2010.xlsx")
OUTPUT_DIR        = Path(".")
DPI               = 180
AÑO_DESDE         = 2015
YLIM_PERCENTIL    = 97
PERM_VERANO       = 70
PERM_INVIERNO     = 30

# Condiciones hidrológicas
CONDICIONES = ["Verano", "Transición", "Invierno"]
COLOR_COND  = {"Verano": "#B71C1C", "Transición": "#2E7D32", "Invierno": "#1565C0"}
MARKER_COND = {"Verano": "s",        "Transición": "^",        "Invierno": "o"}
LSTYLE_COND = {"Verano": ":",        "Transición": "--",        "Invierno": "-"}

# Desplazamiento horizontal leve para evitar superposición entre series
X_OFFSET_COND = {
    "Verano":     -2.2,
    "Transición":  0.0,
    "Invierno":    2.2,
}

# Parámetros con cobertura suficiente
PARAMETROS = {
    "DBO":      {"col": "DEMANDA BIOQUIMICA DE OXIGENO (mg O2/l)",  "unidad": "mg O₂/L"},
    "DQO":      {"col": "DEMANDA QUIMICA DE OXIGENO (mg O2/l)",     "unidad": "mg O₂/L"},
    "OD":       {"col": "OXIGENO DISUELTO (mg O2/l)",               "unidad": "mg O₂/L"},
    "SST":      {"col": "SOLIDOS SUSPENDIDOS TOTALES (mg SS/l)",    "unidad": "mg/L"},
    "NITRATOS": {"col": "NITRATOS (mg N-NO3/l)",                    "unidad": "mg N/L"},
    "P_TOTAL":  {"col": "FOSFORO TOTAL (mg P/l)",                   "unidad": "mg P/L"},
}

# ── Límites físicos por parámetro ──────────────────────────────────────────────
# Valores por encima de estos límites se consideran errores en la fuente.
# NITRATOS: máx real en el río ~5 mg/L; límite conservador = 15 mg/L.
#   57 registros de campañas jul-ago 2022 superan este límite con valores
#   de hasta 150,000,000 mg/L — error confirmado en la base de datos CVC.
#   Los demás parámetros no presentan anomalías en la revisión 2015–2026.
LIMITES_FISICOS = {
    "NITRATOS (mg N-NO3/l)": 15,
}

# Abscisados desde La Balsa (km 0)
# Fuente: Figura 5.11, Informe PMC (Univ. del Valle - CVC)
# ⚠ Pendiente verificación con esquematización del informe de modelación PMC
ABSCISADOS = {
    "PASO DE LA BALSA":   0,
    "PUENTE HORMIGUERO":  50,
    "MEDIACANOA":         145,
    "PUENTE GUAYABAL":    270,
    "LA VICTORIA":        290,
    "ANACARO":            340,
}
NOMBRES_CORTOS = {
    "PASO DE LA BALSA":  "P. Balsa\n(0 km)",
    "PUENTE HORMIGUERO": "Pte. Hormiguero\n(50 km)",
    "MEDIACANOA":        "Mediacanoa\n(145 km)",
    "PUENTE GUAYABAL":   "Guayabal\n(270 km)",
    "LA VICTORIA":       "La Victoria\n(290 km)",
    "ANACARO":           "Anacaro\n(340 km)",
}
ORDEN_EST = list(ABSCISADOS.keys())


# ───────────────────────────────────────────────────────────────────────────────


def leer_caudal_referencia():
    """Lee el Excel de Mediacanoa y calcula umbrales de la curva de duración."""
    wb = load_workbook(CAUDAL_REF_FILE, data_only=True)
    registros = []
    for sheet_name in wb.sheetnames:
        try:
            año = int(sheet_name)
        except ValueError:
            continue
        ws = wb[sheet_name]
        for row in ws.iter_rows(min_row=3, values_only=True):
            if row[0] is None:
                continue
            try:
                dia = int(row[0])
            except Exception:
                continue
            if not 1 <= dia <= 31:
                continue
            for mes_idx, val in enumerate(row[1:13]):
                if isinstance(val, (int, float)):
                    try:
                        fecha = pd.Timestamp(year=año, month=mes_idx + 1, day=dia)
                        registros.append({"FECHA": fecha, "CAUDAL_REF": float(val)})
                    except ValueError:
                        pass

    df_caud = pd.DataFrame(registros)
    datos = np.sort(df_caud["CAUDAL_REF"].values)[::-1]
    permanencia = np.arange(1, len(datos) + 1) / len(datos) * 100
    u_v = float(np.interp(PERM_VERANO,   permanencia, datos))
    u_i = float(np.interp(PERM_INVIERNO, permanencia, datos))
    return df_caud, u_v, u_i


def clasificar_campana(caudal, u_v, u_i) -> str:
    if pd.isna(caudal):   return "Sin clasificar"
    if caudal <= u_v:     return "Verano"
    elif caudal >= u_i:   return "Invierno"
    else:                 return "Transición"


def cargar_datos(df_caud_ref, u_v, u_i) -> pd.DataFrame:
    df = pd.read_excel(CALIDAD_FILE)
    df["FECHA"] = pd.to_datetime(df["FECHA DE MUESTREO"], errors="coerce")
    df["ESTACION_STD"] = (df["ESTACIONES"].str.strip().str.upper()
                            .str.replace(r"\s+", " ", regex=True))
    df = df[df["FECHA"].dt.year >= AÑO_DESDE].copy()
    df = df[df["FECHA"].notna() & df["ESTACION_STD"].isin(ABSCISADOS)].copy()
    df["ABSCISADO"] = df["ESTACION_STD"].map(ABSCISADOS)

    # Convertir parámetros a numérico
    for info in PARAMETROS.values():
        df[info["col"]] = pd.to_numeric(df[info["col"]], errors="coerce")

    # ── Filtro de límites físicos ──────────────────────────────────────────────
    total_filtrados = 0
    for col_name, limite in LIMITES_FISICOS.items():
        if col_name in df.columns:
            mask = df[col_name] > limite
            n = int(mask.sum())
            if n > 0:
                df.loc[mask, col_name] = np.nan
                total_filtrados += n
                print(f"   ⚠  {col_name}: {n} valores > {limite} eliminados "
                      f"(error en fuente CVC — campañas jul/ago 2022)")
    if total_filtrados == 0:
        print("   ✔  Sin valores anómalos detectados")

    # ── Cruzar con caudal de Mediacanoa ───────────────────────────────────────
    df["FECHA_DIA"] = df["FECHA"].dt.normalize()
    df_caud_ref["FECHA_DIA"] = df_caud_ref["FECHA"].dt.normalize()
    df = df.merge(
        df_caud_ref[["FECHA_DIA", "CAUDAL_REF"]].drop_duplicates("FECHA_DIA"),
        on="FECHA_DIA", how="left",
    )
    df["CONDICION"] = df["CAUDAL_REF"].apply(
        lambda q: clasificar_campana(q, u_v, u_i)
    )
    return df


def _label_x():
    return ("Abscisado (km desde La Balsa)  —  "
            "Fuente: PMC Fig. 5.11 (Univ. del Valle - CVC)")


def _nota_pie(fig, u_v, u_i, texto_extra=""):
    nota = (f"Clasificación: Mediacanoa como ref. hidrológica  |  "
            f"Verano ≤{u_v:.0f} m³/s  |  Invierno ≥{u_i:.0f} m³/s  |  "
            f"Fuente: CVC {AÑO_DESDE}–2026")
    if texto_extra:
        nota += f"  |  {texto_extra}"
    fig.text(0.99, 0.005, nota, ha="right", fontsize=6,
             color="#AAAAAA", style="italic")





# ── Gráfica 1: Promedio por condición hidrológica ─────────────────────────────
def graficar_condicion(df, abrev, info, u_v, u_i):
    col    = info["col"]
    unidad = info["unidad"]

    camp_count = df.groupby("CONDICION")["FECHA_DIA"].nunique().to_dict()

    resumen = (df.groupby(["ESTACION_STD", "ABSCISADO", "CONDICION"])[col]
                 .agg(["mean", "std", "count", "min", "max"])
                 .reset_index()
                 .rename(columns={"mean": "MEDIA", "std": "STD",
                                  "count": "N", "min": "MIN", "max": "MAX"}))
    resumen = resumen[resumen["N"] >= 2]
    resumen = resumen[resumen["CONDICION"].isin(CONDICIONES)]

    if resumen.empty:
        print(f"  ⚠  {abrev} condición: sin datos suficientes")
        return

    y_cap = float(np.nanpercentile(df[col].dropna(), YLIM_PERCENTIL))

    fig, ax = plt.subplots(figsize=(12.6, 5.8))
    fig.patch.set_facecolor("#F7F9FC")
    ax.set_facecolor("#F7F9FC")

    puntos = []
    for cond in CONDICIONES:
        sub = resumen[resumen["CONDICION"] == cond].sort_values("ABSCISADO")
        if sub.empty:
            continue

        c        = COLOR_COND[cond]
        n        = camp_count.get(cond, 0)
        x_offset = X_OFFSET_COND.get(cond, 0.0)
        x_plot   = sub["ABSCISADO"] + x_offset

        ax.plot(x_plot, sub["MEDIA"],
                color=c, linestyle=LSTYLE_COND[cond], linewidth=2,
                marker=MARKER_COND[cond], markersize=7,
                markerfacecolor="white", markeredgecolor=c,
                markeredgewidth=2, zorder=4,
                label=f"{cond}  (n={n} campañas)")

        ax.fill_between(x_plot,
                        sub["MIN"].clip(lower=0),
                        sub["MAX"].clip(upper=y_cap),
                        alpha=0.10, color=c, zorder=2)

        for (_, r), x_val in zip(sub.iterrows(), x_plot):
            puntos.append((x_val, r["MEDIA"], f"{r['MEDIA']:.1f}", c, cond))

    # Etiquetas: posición vertical escalonada + pequeño offset horizontal
    # para evitar superposición cuando varios valores son cercanos
    df_lbl = pd.DataFrame(puntos, columns=["x", "y", "texto", "color", "cond"])
    # Offset horizontal fijo por condición (alineado con X_OFFSET_COND)
    lbl_x_off = {"Verano": -2.2, "Transición": 0.0, "Invierno": 2.2}
    # Offset vertical fijo por condición para separar etiquetas cercanas
    lbl_y_off = {"Verano": 0.04, "Transición": 0.08, "Invierno": -0.04}

    for _, grp in df_lbl.groupby(df_lbl["x"].round(0)):
        grp = grp.sort_values("y").reset_index(drop=True)
        y_range = y_cap
        for _, row in grp.iterrows():
            y_off_frac = lbl_y_off.get(row["cond"], 0.04)
            y_pos = row["y"] + y_range * y_off_frac
            x_pos = row["x"] + lbl_x_off.get(row["cond"], 0)
            ax.text(x_pos, y_pos, row["texto"],
                    ha="center", va="bottom",
                    fontsize=7.5, color=row["color"],
                    fontweight="bold", clip_on=True, zorder=5)

    for km in ABSCISADOS.values():
        ax.axvline(km, color="#DDDDDD", linewidth=0.6, zorder=1)

    ax.set_xticks(list(ABSCISADOS.values()))
    ax.set_xticklabels([NOMBRES_CORTOS[e] for e in ORDEN_EST],
                       fontsize=8.5, ha="center", linespacing=1.1)

    ax.set_xlim(-18, 358)
    ax.set_ylim(0, y_cap * 1.36)
    ax.set_ylabel(f"{abrev} ({unidad})", fontsize=10, labelpad=8)
    ax.set_xlabel(_label_x(), fontsize=8, labelpad=8, color="#666666")
    ax.tick_params(axis="x", pad=4)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}"))
    ax.grid(axis="y", linestyle="--", alpha=0.35, zorder=0)
    ax.legend(fontsize=9, title="Condición hidrológica",
              title_fontsize=9, loc="best", framealpha=0.85)

    n_sin = camp_count.get("Sin clasificar", 0)
    ax.set_title(
        f"Perfil Longitudinal de {abrev} — Río Cauca  ({AÑO_DESDE}–2026)\n"
        f"Por condición hidrológica  |  Ref: Mediacanoa  |  "
        f"Banda = rango máximo–mínimo",
        fontsize=11, fontweight="bold", pad=12, color="#1A1A2E")

    n_out = int((df[col].dropna() > y_cap).sum())
    _nota_pie(fig, u_v, u_i, f"✂ {n_out} outliers fuera del eje" if n_out else "")
    fig.tight_layout()
    fig.subplots_adjust(bottom=0.16)
    out = OUTPUT_DIR / f"perfil_{abrev}_condicion.png"
    fig.savefig(out, dpi=DPI, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"  ✔  {abrev:10s} condición → {out.name}")


# ── Gráfica 2: Box plot por condición ─────────────────────────────────────────
def graficar_boxplot(df, abrev, info, u_v, u_i):
    col    = info["col"]
    unidad = info["unidad"]

    camp_count = df.groupby("CONDICION")["FECHA_DIA"].nunique().to_dict()
    df_p = df[df["CONDICION"].isin(CONDICIONES)][
        ["ESTACION_STD", "CONDICION", col]].dropna(subset=[col])

    if df_p.empty:
        print(f"  ⚠  {abrev} boxplot: sin datos")
        return

    y_cap   = float(np.nanpercentile(df_p[col], YLIM_PERCENTIL))
    n_out   = int((df_p[col] > y_cap).sum())
    offsets = [-0.8, 0.0, 0.8]

    fig, ax = plt.subplots(figsize=(13, 5))
    fig.patch.set_facecolor("#F7F9FC")
    ax.set_facecolor("#F7F9FC")

    posiciones_xtick = []
    labels_xtick     = []

    for i, est in enumerate(ORDEN_EST):
        x_base  = i * 3
        posiciones_xtick.append(x_base)
        n_total = len(df_p[df_p["ESTACION_STD"] == est])
        labels_xtick.append(f"{NOMBRES_CORTOS[est]}\nn={n_total} datos")

        for cond, offset in zip(CONDICIONES, offsets):
            datos = df_p[(df_p["ESTACION_STD"] == est) &
                         (df_p["CONDICION"]    == cond)][col].dropna()
            if len(datos) < 3:
                continue
            c = COLOR_COND[cond]
            ax.boxplot(datos.clip(upper=y_cap),
                       positions=[x_base + offset],
                       widths=0.6, patch_artist=True, showfliers=True,
                       flierprops=dict(marker="+", markersize=4,
                                       markeredgecolor=c, alpha=0.5,
                                       linestyle="none"),
                       medianprops=dict(color="white", linewidth=2.5),
                       whiskerprops=dict(color=c, linewidth=1.2),
                       capprops=dict(color=c, linewidth=1.5),
                       boxprops=dict(facecolor=c, alpha=0.60, linewidth=0),
                       zorder=3)

        if i < len(ORDEN_EST) - 1:
            ax.axvline(x_base + 1.5, color="#DDDDDD",
                       linewidth=0.8, linestyle="-", zorder=1)

    handles = [
        mpatches.Patch(facecolor=COLOR_COND[c], alpha=0.7,
                       label=f"{c}  (n={camp_count.get(c, 0)} campañas)")
        for c in CONDICIONES
    ]
    ax.legend(handles=handles, title="Condición hidrológica",
              fontsize=9, title_fontsize=9, loc="best", framealpha=0.85)

    ax.set_xticks(posiciones_xtick)
    ax.set_xticklabels(labels_xtick, fontsize=7.5, ha="center")

    ax.set_xlim(-1.8, (len(ORDEN_EST) - 1) * 3 + 1.8)
    ax.set_ylim(0, y_cap * 1.10)
    ax.set_ylabel(f"{abrev} ({unidad})", fontsize=10, labelpad=8)
    ax.set_xlabel(_label_x(), fontsize=8, labelpad=8, color="#666666")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}"))
    ax.grid(axis="y", linestyle="--", alpha=0.35, zorder=0)
    ax.set_title(
        f"Distribución de {abrev} por Condición Hidrológica — Río Cauca  "
        f"({AÑO_DESDE}–2026)\n"
        f"Box plot por estación  |  Ref: Mediacanoa  |  "
        f"Línea blanca = mediana  |  Caja = IQR",
        fontsize=11, fontweight="bold", pad=12, color="#1A1A2E")
    _nota_pie(fig, u_v, u_i, f"✂ {n_out} outliers fuera del eje" if n_out else "")
    fig.tight_layout()
    out = OUTPUT_DIR / f"perfil_{abrev}_boxplot.png"
    fig.savefig(out, dpi=DPI, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"  ✔  {abrev:10s} boxplot   → {out.name}  "
          f"({'✂ ' + str(n_out) + ' outliers' if n_out else 'sin recorte'})")


# ── Main ───────────────────────────────────────────────────────────────────────
def main():
    if not CALIDAD_FILE.exists():
        raise FileNotFoundError(f"No se encontró '{CALIDAD_FILE}'")
    if not CAUDAL_REF_FILE.exists():
        raise FileNotFoundError(
            f"No se encontró '{CAUDAL_REF_FILE}'\n"
            f"Asegúrate de tener la carpeta MEDICANOA con caudal_2010.xlsx")

    print(f"📄 Leyendo caudales de referencia: {CAUDAL_REF_FILE} ...")
    df_caud, u_v, u_i = leer_caudal_referencia()
    print(f"   → {len(df_caud):,} caudales diarios")
    print(f"   → Umbral Verano     ≤ {u_v:.1f} m³/s")
    print(f"   → Umbral Transición:  {u_v:.1f} – {u_i:.1f} m³/s")
    print(f"   → Umbral Invierno   ≥ {u_i:.1f} m³/s")

    print(f"\n📄 Leyendo calidad del agua: {CALIDAD_FILE} ...")
    df = cargar_datos(df_caud, u_v, u_i)

    camp_class = df.groupby("CONDICION")["FECHA_DIA"].nunique()
    print(f"\n   Clasificación de {df['FECHA_DIA'].nunique()} campañas:")
    for cond, n in camp_class.items():
        print(f"     {cond:15s}: {n} campañas")

    print(f"\n📊 Generando gráficas:\n")
    for abrev, info in PARAMETROS.items():
        graficar_condicion(df, abrev, info, u_v, u_i)
        graficar_boxplot(df, abrev, info, u_v, u_i)

    print(f"\n✅ Listo. {len(PARAMETROS) * 2} gráficas guardadas.")


if __name__ == "__main__":
    main()

📄 Leyendo caudales de referencia: MEDICANOA\caudal_2010.xlsx ...
   → 5,600 caudales diarios
   → Umbral Verano     ≤ 196.2 m³/s
   → Umbral Transición:  196.2 – 406.2 m³/s
   → Umbral Invierno   ≥ 406.2 m³/s

📄 Leyendo calidad del agua: Calidad_del_agua_del_Rio_Cauca_20260501.xlsx ...
   ⚠  NITRATOS (mg N-NO3/l): 18 valores > 15 eliminados (error en fuente CVC — campañas jul/ago 2022)

   Clasificación de 30 campañas:
     Invierno       : 5 campañas
     Sin clasificar : 1 campañas
     Transición     : 14 campañas
     Verano         : 10 campañas

📊 Generando gráficas:

  ✔  DBO        condición → perfil_DBO_condicion.png
  ✔  DBO        boxplot   → perfil_DBO_boxplot.png  (✂ 5 outliers)
  ✔  DQO        condición → perfil_DQO_condicion.png
  ✔  DQO        boxplot   → perfil_DQO_boxplot.png  (✂ 4 outliers)
  ✔  OD         condición → perfil_OD_condicion.png
  ✔  OD         boxplot   → perfil_OD_boxplot.png  (✂ 5 outliers)
  ✔  SST        condición → perfil_SST_condicion.png
  ✔  SST

In [ ]:
import pandas as pd
from datetime import datetime

# ── Configuración ──────────────────────────────────────────────
CALIDAD_FILE = "Calidad_del_agua_del_Rio_Cauca_20260309.xlsx"
AÑOS_ATRAS   = 11
# ───────────────────────────────────────────────────────────────

df = pd.read_excel(CALIDAD_FILE)
df['FECHA'] = pd.to_datetime(df['FECHA DE MUESTREO'], errors='coerce')
df['ESTACIONES'] = (df['ESTACIONES'].str.strip()
                                    .str.upper()
                                    .str.replace(r'\s+', ' ', regex=True))

año_inicio = datetime.now().year - AÑOS_ATRAS
df_10 = df[df['FECHA'].dt.year >= año_inicio].copy()

print(f"📅 Período analizado: {año_inicio} – {datetime.now().year}")
print(f"📊 Total registros  : {len(df_10)}")
print(f"📍 Estaciones únicas: {df_10['ESTACIONES'].nunique()}\n")

# Registros por estación con rango de fechas
resumen = (df_10.groupby('ESTACIONES')
                .agg(
                    Registros  = ('FECHA', 'count'),
                    Fecha_min  = ('FECHA', 'min'),
                    Fecha_max  = ('FECHA', 'max'),
                )
                .sort_values('Registros', ascending=False)
                .reset_index())

resumen['Fecha_min'] = resumen['Fecha_min'].dt.strftime('%Y-%m-%d')
resumen['Fecha_max'] = resumen['Fecha_max'].dt.strftime('%Y-%m-%d')
resumen.columns = ['Estación', 'Registros', 'Primer registro', 'Último registro']

print(resumen.to_string(index=False))

📅 Período analizado: 2015 – 2026
📊 Total registros  : 570
📍 Estaciones únicas: 20

             Estación  Registros Primer registro Último registro
              ANACARO         30      2015-03-18      2024-12-06
     PASO DE LA TORRE         30      2015-03-18      2024-12-06
                VIJES         30      2015-03-18      2024-12-06
              RIOFRIO         30      2015-03-18      2024-12-06
        PUERTO ISAACS         30      2015-03-18      2024-12-06
   PUENTE LA VIRGINIA         30      2015-03-18      2024-12-06
    PUENTE HORMIGUERO         30      2015-03-18      2024-12-06
      PUENTE GUAYABAL         30      2015-03-18      2024-12-06
    PASO DEL COMERCIO         30      2015-03-18      2024-12-06
     PASO DE LA BOLSA         30      2015-03-18      2024-12-06
     PASO DE LA BALSA         30      2015-03-18      2024-12-06
           MEDIACANOA         30      2015-03-18      2024-12-06
          LA VICTORIA         30      2015-03-18      2024-12-06
       

In [ ]:
'''
# =============================================================
# PASO 1: Filtrar ríos tributarios desde shapefile local
# Fuente: Tributarios.shp (exportado del servidor CVC)
# ArcMap 10.8 / Python 2.7
# =============================================================

import arcpy
import os

WORKSPACE    = r"C:\Users\ASUS\Desktop\Work\Proyecto - Corredor Biológico\Fase I\Cartografía"
SHP_FUENTE   = os.path.join(WORKSPACE, "Tributarios.shp")
SHP_RIOS     = os.path.join(WORKSPACE, "rios_tributarios.shp")
CAMPO_NOMBRE = "NOM1_DRENA"

RIOS_INTERES = [
    "Rio Palo",
    "Rio Desbaratado",
    "Rio Fraile",
    "Rio Bolo",
    "Rio Amaime",
    "Rio Zabaletas",
    "Rio Guabas",
    "Rio Guadalajara",
    "Rio Riofrio",
    "Rio Tulua",
    "Rio Bugalagrande",
    "Rio La Paila",
    "Rio Risaralda",
]

arcpy.env.workspace       = WORKSPACE
arcpy.env.overwriteOutput = True

# -------------------------------------------------------
# 1. Verificar shapefile y mostrar campos y valores
# -------------------------------------------------------
print("Leyendo {}...".format(SHP_FUENTE))
n_total = int(arcpy.GetCount_management(SHP_FUENTE).getOutput(0))
print("-> {} segmentos en Tributarios.shp".format(n_total))

campos = [f.name for f in arcpy.ListFields(SHP_FUENTE)]
print("Campos: {}".format(campos))

print("\nValores unicos en '{}':".format(CAMPO_NOMBRE))
vals = set()
with arcpy.da.SearchCursor(SHP_FUENTE, [CAMPO_NOMBRE]) as cur:
    for row in cur:
        if row[0]: vals.add(row[0])
for v in sorted(vals):
    print("  {}".format(repr(v)))

# -------------------------------------------------------
# 2. Filtrar los 13 tributarios con LIKE
# -------------------------------------------------------
print("\nFiltrando tributarios...")

where_rios = " OR ".join([
    "{} LIKE '%{}%'".format(CAMPO_NOMBRE, r)
    for r in RIOS_INTERES
])

arcpy.MakeFeatureLayer_management(SHP_FUENTE, "rios_lyr")
arcpy.SelectLayerByAttribute_management("rios_lyr", "NEW_SELECTION", where_rios)
arcpy.CopyFeatures_management("rios_lyr", SHP_RIOS)
arcpy.Delete_management("rios_lyr")

n_rios = int(arcpy.GetCount_management(SHP_RIOS).getOutput(0))
print("-> {} segmentos seleccionados".format(n_rios))

# -------------------------------------------------------
# 3. Verificar cuáles se encontraron
# -------------------------------------------------------
encontrados = set()
with arcpy.da.SearchCursor(SHP_RIOS, [CAMPO_NOMBRE]) as cur:
    for row in cur:
        if row[0]: encontrados.add(row[0].strip())

print("\nEstado por rio:")
for r in RIOS_INTERES:
    ok = any(r in v.upper() for v in encontrados)
    print("  [{}] {}".format("OK" if ok else "*** FALTA ***", r))

print("\n-> rios_tributarios.shp listo")
print("   Segmentos: {}  |  Rios unicos: {}".format(n_rios, len(encontrados)))
'''